# Getting Started with QuScope

Welcome to QuScope! This tutorial will guide you through the basics of using QuScope for quantum-enhanced microscopy analysis.

## What is QuScope?

QuScope is a Python package that applies quantum computing algorithms to electron microscopy image processing and EELS analysis. It's built on Qiskit and provides:

- Quantum CTEM (Conventional Transmission Electron Microscopy) simulation
- Quantum image encoding and processing
- EELS quantum analysis
- Integration with IBM Quantum hardware

## Installation

If you haven't already installed QuScope:

```bash
pip install quscope
```

## 1. Basic Imports and Setup

Let's start by importing the necessary modules:

In [ ]:
import quscope
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit

# QuScope quantum CTEM modules
from quscope.quantum_ctem.backends import get_backend
from quscope.quantum_ctem.materials import get_material

print(f"QuScope version: {quscope.__version__}")
print("Setup complete!")

## 2. Working with Quantum Backends

QuScope provides a unified backend abstraction for quantum simulators and real quantum hardware.

In [ ]:
# Get a simulator backend (no IBM credentials needed)
backend = get_backend('simulator')
backend.connect()

print(f"Backend type: {backend.__class__.__name__}")
print("Backend initialized successfully")

# Create a simple Bell state circuit
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])

print("\nBell state circuit:")
print(qc)

# Execute the circuit using the backend.run() method
from quscope.quantum_ctem.backends import BackendConfig

config = BackendConfig(shots=1000)
result = backend.run(qc, config)
counts = result.counts

print("\nMeasurement results:")
for state, count in sorted(counts.items()):
    print(f"  |{state}⟩: {count} times ({count/10:.1f}%)")

In [ ]:
# Visualize the results
states = list(counts.keys())
values = list(counts.values())

plt.figure(figsize=(8, 6))
plt.bar(states, values, color='steelblue', alpha=0.7, edgecolor='black')
plt.xlabel('Measurement Outcome', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Bell State Measurement Results', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nTotal shots: {sum(values)}")
print(f"Outcome |00⟩: {counts.get('00', 0)} times")
print(f"Outcome |11⟩: {counts.get('11', 0)} times")

## 3. Loading Materials

QuScope includes pre-defined materials with their atomic structures and scattering properties.

In [ ]:
# Load MoS2 material
mos2 = get_material('mos2')

print(f"Material: {mos2.name}")
print(f"Formula: {mos2.formula}")
print(f"\nLattice parameters:")
print(f"  a = {mos2.parameters.a:.3f} Å")
print(f"  c = {mos2.parameters.c:.3f} Å")
print(f"\nElements: {mos2.parameters.elements}")
print(f"Space group: {mos2.parameters.space_group}")
print(f"Typical thickness: {mos2.parameters.typical_thickness:.1f} Å")

In [ ]:
# Load Graphene material
graphene = get_material('graphene')

print(f"Material: {graphene.name}")
print(f"Formula: {graphene.formula}")
print(f"\nLattice parameters:")
print(f"  a = {graphene.parameters.a:.3f} Å")
print(f"  c = {graphene.parameters.c:.3f} Å")
print(f"\nElements: {graphene.parameters.elements}")
print(f"Space group: {graphene.parameters.space_group}")
print(f"Typical thickness: {graphene.parameters.typical_thickness:.1f} Å")

## 4. Simple Quantum Image Encoding

Let's encode a small image into a quantum circuit.

In [ ]:
# Quantum Wave Function Encoding
from quscope.quantum_ctem.quantum_encoding import QuantumWaveEncoder

# Create a simple 4x4 test image (representing a 2D wavefunction)
test_image = np.array([
    [0.0, 0.2, 0.8, 1.0],
    [0.1, 0.3, 0.7, 0.9],
    [0.2, 0.4, 0.6, 0.8],
    [0.3, 0.5, 0.5, 0.7]
])

# Normalize the image
test_image_normalized = test_image / np.linalg.norm(test_image)

print("Test wavefunction (4×4):")
print(test_image_normalized)

# Visualize the wavefunction
plt.figure(figsize=(6, 6))
plt.imshow(test_image_normalized, cmap='viridis', interpolation='nearest')
plt.colorbar(label='Amplitude')
plt.title('4×4 Test Wavefunction')
plt.show()

In [ ]:
# Encode wavefunction to quantum circuit
encoder = QuantumWaveEncoder(grid_size=4)  # 4x4 grid requires 2 qubits per dimension

# Use amplitude encoding
circuit = encoder.encode_amplitude(test_image_normalized)

print(f"Quantum encoded circuit:")
print(f"  Qubits: {circuit.num_qubits}")
print(f"  Depth: {circuit.depth()}")
print(f"  Gates: {circuit.size()}")

# Visualize the circuit structure
print(f"\nCircuit structure:")
print(circuit)

## 5. Quantum State Analysis

Let's analyze the quantum state created by our encoding.

In [ ]:
from qiskit.quantum_info import Statevector

# Get the statevector from the encoded circuit (only works with simulators)
statevector = Statevector.from_instruction(circuit)

# Get probabilities
probabilities = statevector.probabilities()

# Compare with flattened, normalized wavefunction
flattened_normalized = test_image_normalized.flatten()

print("Quantum State Analysis:")
print("=" * 60)
print(f"Total probability: {np.sum(probabilities):.6f}")
print(f"\nFirst 8 basis states and their amplitudes:")

for i in range(min(8, len(probabilities))):
    amplitude = np.sqrt(probabilities[i])
    expected = flattened_normalized[i]
    print(f"  |{i:04b}⟩: amplitude = {amplitude:.4f}, expected = {expected:.4f}")